### Cell 1: Install dependencies

In [1]:
!pip install -q sentence-transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00:00:0100:01


### Cell 2: Imports and helpers

In [3]:
import gc
import time

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

from huggingface_hub import notebook_login

notebook_login()

MODEL_NAME = "google/embeddinggemma-300m"

# We'll store results here as we go
results = {}


def get_vram_mb() -> float:
    """Current GPU VRAM allocation in MB."""
    return torch.cuda.memory_allocated(0) / (1024 * 1024)


def cleanup():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def benchmark(model, sentences, n_runs=10):
    """Generate embeddings, measure VRAM and inference time.

    Returns a dict with vram_mb, embeddings, mean_time_ms, std_time_ms.
    """
    vram = get_vram_mb()

    # Warmup — first encode always slower (CUDA kernel compilation etc.)
    _ = model.encode(sentences[:2], show_progress_bar=False)

    # Timed runs
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        embeddings = model.encode(
            sentences, batch_size=32, show_progress_bar=False, convert_to_numpy=True
        )
        torch.cuda.synchronize()  # wait for GPU to finish
        times.append(time.perf_counter() - start)

    return {
        "vram_mb": vram,
        "embeddings": embeddings,
        "mean_time_ms": np.mean(times) * 1000,
        "std_time_ms": np.std(times) * 1000,
    }


print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


GPU: Tesla T4
VRAM total: 14.6 GB


### Cell 3: Test sentences

In this case, these are a mix of real excerpts from SEC filings and short search queries.

In [4]:
# Want to test that quantisation preserves semantic nuance across different types of content.
test_sentences = [
    # across different content types and lengths from real SEC filings
    "•developments or disputes concerning our intellectual property or other proprietary rights;",
    "We are subject to claims, lawsuits, regulatory and government inquiries and investigations, other proceedings, and orders involving competition, intellectual property, data privacy and security, tax and related compliance, labor and employment, commercial disputes, content generated by our users, goods and services offered by advertisers or publishers using our platforms, personal injury, and other matters.",
    "We are exposed to financial market risks, including changes in foreign currency exchange rates, interest rates, and equity investment risks.",
    "Our marketable and non-marketable equity securities are subject to a wide variety of market-related risks that could substantially reduce or increase the fair value of our holdings.",
    "Innovations in our products and services could also result in changes to user and customer behavior and affect our revenue trends. These endeavors involve significant risks and uncertainties, including diversion of resources and management attention from current operations, different monetization models, and the use of alternative investment, governance, or compensation structures that may fail to adequately align incentives across the company or otherwise accomplish their objectives.",
    "From time to time, we acquire or invest in businesses, content, and technologies that support our business. The risks associated with such acquisitions or investments (some of which may be unforeseen) include the difficulty of integrating solutions, operations, and personnel; inheriting liabilities and exposure to litigation; failure to realize anticipated benefits and expected synergies; and diversion of management’s time and attention, among other acquisition-related risks.",
    "We view our employees and our culture as key to our success.  As of December 31, 2024, we had approximately 14,000 full-time employees.  Of these, approximately 9,600 (69%) were located in the United States and Canada, 2,200 (16%) in Europe, Middle East, and Africa, 600 (4%) in Latin America and 1,600 (11%) in Asia-Pacific.  We also have a number of employees engaged in content production, some of whom are part-time or temporary, and whose numbers fluctuate throughout the year and may be covered by collective bargaining agreements",
    "As of December 27, 2025, we had approximately 31,000 employees in our global workforce. We believe we are at our best when our culture of innovation and people from all kinds of backgrounds work together in an engaging and open environment. Areas of focus for us include:",
    "Volatility in our stock price could adversely affect our business and financing opportunities and force us to increase our cash compensation to employees or grant larger stock awards than we have historically, which could hurt our operating results or reduce the percentage ownership of our existing stockholders, or both.",
    "The semiconductor industry is highly cyclical and has experienced severe downturns.",
    "We believe trade dynamics and geopolitics are disrupting and reshaping global supply chains and affecting customer order behavior. Our global manufacturing capabilities enable us to support our customers’ needs. The overall semiconductor market recovery is continuing, though at a slower pace than prior upturns, likely related to broader macroeconomic dynamics and overall uncertainty.",
    "The “semiconductor cycle” refers to the ebb and flow of supply and demand and the building and depleting of inventories. The semiconductor market historically has been characterized by periods of tight supply caused by strengthening demand and/or insufficient manufacturing capacity, followed by periods of surplus inventory caused by weakening demand and/or excess manufacturing capacity. These are typically referred to as upturns and downturns in the semiconductor cycle. Semiconductor cycles are affected by the significant time and money required to build and maintain semiconductor manufacturing facilities.",
    "The terms of certain of our debt facilities contain, and any of our other future debt agreements may contain, covenant restrictions that may limit our ability to operate our business, including restrictions on our and/or our subsidiaries’ ability to, among other things, incur additional debt or create liens. In addition, under certain circumstances we are required to maintain a certain amount of liquidity. As a result of these covenants, our ability to respond to changes in business and economic conditions and engage in beneficial transactions, including to obtain additional financing as needed, may be restricted. Furthermore, our failure to comply with our debt covenants could result in a default under our debt agreements, which could permit the holders to accelerate our obligation to repay the debt. If any of our debt is accelerated, we may not have sufficient funds available to repay it.",
    "We have financial instruments that are subject to interest rate risk, principally fixed-rate debt obligations. The investors in our fixed-rate debt obligations do not generally have the right to demand we pay off these obligations prior to maturity. Therefore, exposure to interest rate risk is not believed to be material for our fixed-rate debt.",
    "•changes to U.S. and non-U.S. government policies, including sourcing restrictions, requirements to expend a portion of program funds locally and governmental industrial cooperation or participation requirements;",
    "•sanctions, import and export controls, other market access barriers, political unrest, geopolitical tensions, changes in regimes, or armed conflict (such as ongoing conflicts in the Middle East and Ukraine), any of which may affect our business continuity, increase our operating costs, limit demand for our products and services, limit our ability to source components or final products, or prevent or impede us from operating in certain jurisdictions, complying with local laws, or offering products or services;",
    "•an evolving foreign policy landscape that could harm our revenues and could subject us to litigation, new regulatory costs and challenges (including new customer requirements), uncertainty regarding regulatory outcomes, and other liabilities under local laws that may not offer due process or clear legal precedent;",
    "As a corporation with an international presence, we are subject to risks and uncertainties caused by significant events with macroeconomic impacts, including, but not limited to, geopolitical tensions, heightened interest rates, monetary policy changes,  foreign currency fluctuations, and the imposition of tariffs or other impacts on trade relations. Additionally, these macroeconomic impacts have disrupted, and may continue to disrupt, the operations of our customers and prospective customers. We continuously monitor the direct and indirect impacts of these circumstances on our business and financial results, as well as the overall global economy and geopolitical landscape.",
    "•changes to the content or application of third-party policies that limit our ability to deliver, target, or measure the effectiveness of advertising, including changes by mobile operating system and browser providers such as Apple and Google;",
    "We continue adapting our strategy to meet our liquidity and risk objectives, such as investing in U.S. government securities and other investments, investing in autonomy, further vertically integrating our supply chain, expanding our product roadmap and providing financing options to our customers.",
    "As a result of rapidly evolving trade policy, uncertainty in the automotive and energy markets continues to increase, posing risks to our global supply chain and cost structure which could have a meaningfully adverse impact on demand for our products and our profitability. The current tariff regime will have a relatively larger impact on our energy generation and storage business compared to our automotive business. While we prepare for near-term challenges to our business under current policies, we are focused on long-term growth opportunities as we continue to make prudent investments.",
    "Our effective tax rate is subject to significant variation due to several factors, including variability in our pre-tax and taxable income and loss and the mix of jurisdictions to which they relate, intercompany transactions, the applicability of special tax regimes, changes in how we do business, acquisitions, investments, developments in tax controversies, changes in our stock price, changes in our deferred tax assets and liabilities and their valuation, foreign currency gains (losses), changes in statutes, regulations, case law, and administrative practices, principles, and interpretations related to tax, including changes to the global tax framework, competition, and other laws and accounting rules in various jurisdictions, and relative changes of expenses or losses for which tax benefits are not recognized. Our effective tax rate can be more or less volatile based on the amount of pre-tax income or loss. For example, the impact of discrete items and non-deductible expenses on our effective tax rate is greater when our pre-tax income is lower. In addition, we record valuation allowances against deferred tax assets when there is uncertainty about our ability to generate future income in relevant jurisdictions.",
    "Our effective tax rates for the first three and nine months of 2025 and 2024 as compared to the U.S. federal statutory rate of 21% were primarily impacted by the mix of our jurisdictional earnings subject to different tax rates, valuation allowances on our deferred tax assets and benefits from our U.S. research and development credits, and manufacturing production credits.",
    # Short search queries
    "How is the company addressing the transition to a low-carbon economy in its long-term strategy?",
    "What are the legal issues?",
    "What are the risks?",
    "What are the geopolitical risks for company?",
    "How many employees does the company have?",
    "Executive compensation and stock options",
    "Semiconductor supply chain dependency",
    "Debt covenant compliance risk",
    "revenue recognition policy changes",
    "Explain the primary reasons for the difference between the company’s effective tax rate and the statutory federal rate."
]

print(f"{len(test_sentences)} test sentences ready")

33 test sentences ready


### Cell 4: FP32 Baseline

In [5]:
# FP32 is the original precision — this is our ground truth.
# Every other variant will be compared against these embeddings.
cleanup()
vram_before = get_vram_mb()

model_fp32 = SentenceTransformer(MODEL_NAME, device="cuda")

res = benchmark(model_fp32, test_sentences)
res["vram_mb"] -= vram_before  # isolate model-only VRAM
results["FP32"] = res

print(f"VRAM (model only): {res['vram_mb']:.1f} MB")
print(f"Inference time:    {res['mean_time_ms']:.1f} ± {res['std_time_ms']:.1f} ms")
print(f"Embedding shape:   {res['embeddings'].shape}")
print(f"Embedding dtype:   {res['embeddings'].dtype}")

del model_fp32
cleanup()

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

VRAM (model only): 1177.7 MB
Inference time:    595.2 ± 73.2 ms
Embedding shape:   (33, 768)
Embedding dtype:   float32


### Cell 5: BF16 (half precision)

In [6]:
# BF16 instead of FP16 for stability.
# BF16 (Brain Float 16) is better than FP16 for transformers:
# - Same range as FP32 (±3.4e38), so no overflow/underflow
# - 8-bit precision instead of 16-bit, so smaller model
# - Used by default in many transformer training runs
# The "Brain" part is Google's name for it (they developed it).

cleanup()
vram_before = get_vram_mb()

model_bf16 = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
    model_kwargs={"torch_dtype": torch.bfloat16},
)

res = benchmark(model_bf16, test_sentences)
res["vram_mb"] -= vram_before
results["BF16"] = res

print(f"VRAM (model only): {res['vram_mb']:.1f} MB")
print(f"Inference time:    {res['mean_time_ms']:.1f} ± {res['std_time_ms']:.1f} ms")

del model_bf16
cleanup()

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

VRAM (model only): 592.0 MB
Inference time:    774.8 ± 44.7 ms


### Cell 6: INT8 (bitsandbytes)

INT8 quantisation via bitsandbytes uses the LLM.int8() algorithm from the paper "LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale" (Dettmers et al., 2022).

Key idea: most weight values are small and can be safely stored in INT8 (256 discrete levels). But a small fraction of "outlier" features have much larger magnitudes — quantising those to INT8 would destroy information. LLM.int8() identifies these outliers at runtime and processes them in FP16, while the rest stays INT8. This mixed-precision approach preserves quality remarkably well.

In [7]:
cleanup()
vram_before = get_vram_mb()

bnb_config_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = SentenceTransformer(
    MODEL_NAME,
    model_kwargs={
        "quantization_config": bnb_config_int8,
        # bitsandbytes places the model on GPU automatically;
        # device_map tells transformers not to fight over placement
        "device_map": "auto",
    },
)

res = benchmark(model_int8, test_sentences)
res["vram_mb"] -= vram_before
results["INT8"] = res

print(f"VRAM (model only): {res['vram_mb']:.1f} MB")
print(f"Inference time:    {res['mean_time_ms']:.1f} ± {res['std_time_ms']:.1f} ms")

del model_int8
cleanup()

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


VRAM (model only): 883.6 MB
Inference time:    872.1 ± 52.4 ms


### Cell 7: NF4 4-bit (bitsandbytes)

NF4 (Normal Float 4-bit) is from the QLoRA paper (Dettmers et al., 2023).

Key idea: neural network weights tend to follow a normal distribution. NF4 defines 16 quantisation levels that are optimally spaced for a standard normal distribution (denser near zero, sparser at tails). This gives better information retention than naive uniform 4-bit quantisation.

bnb_4bit_compute_dtype=float16 means: weights are STORED in 4-bit, but DEQUANTISED to FP16 for actual matrix multiplications. This keeps computation accurate while storage is compressed.

In [8]:
cleanup()
vram_before = get_vram_mb()

bnb_config_nf4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model_nf4 = SentenceTransformer(
    MODEL_NAME,
    model_kwargs={
        "quantization_config": bnb_config_nf4,
        "device_map": "auto",
    },
)

res = benchmark(model_nf4, test_sentences)
res["vram_mb"] -= vram_before
results["NF4"] = res

print(f"VRAM (model only): {res['vram_mb']:.1f} MB")
print(f"Inference time:    {res['mean_time_ms']:.1f} ± {res['std_time_ms']:.1f} ms")

del model_nf4
cleanup()

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

VRAM (model only): 840.8 MB
Inference time:    447.1 ± 28.1 ms


### Cell 8: Quality comparison

The critical question: do the quantised embeddings still capture the same semantic relationships?

We measure this two ways:
1. Per-sentence cosine similarity vs FP32 (closer to 1.0 = better)
2. Ranking preservation: does a query return results in the same order?


In [9]:
from numpy.linalg import norm


def cosine_sim(a, b):
    """Cosine similarity between two vectors."""
    return np.dot(a, b) / (norm(a) * norm(b))


fp32_embs = results["FP32"]["embeddings"]

# --- Per-sentence similarity vs FP32 ---
print("Per-sentence cosine similarity vs FP32 baseline")
print("=" * 72)

for variant in ["BF16", "INT8", "NF4"]:  # Changed from FP16 to BF16
    embs = results[variant]["embeddings"]
    sims = [cosine_sim(fp32_embs[i], embs[i]) for i in range(len(fp32_embs))]

    print(f"\n{variant}:")
    for i, sim in enumerate(sims):
        marker = "✓" if sim > 0.99 else "~" if sim > 0.95 else "✗"
        print(f"  {marker} {sim:.6f}  {test_sentences[i][:65]}...")
    print(f"  ── Mean: {np.mean(sims):.6f}  Min: {np.min(sims):.6f}")

# --- Ranking preservation ---
query_idx = 8
if len(test_sentences) <= query_idx:
    print(f"\nWarning: test_sentences has only {len(test_sentences)} items, skipping ranking test")
else:
    doc_indices = [i for i in range(len(test_sentences)) if i != query_idx]
    print(f"\nRanking preservation test")
    print("=" * 72)
    print(f"Query: \"{test_sentences[query_idx]}\"\n")

    for variant in ["FP32", "BF16", "INT8", "NF4"]:  # Changed from FP16 to BF16
        embs = results[variant]["embeddings"]
        sims = [(i, cosine_sim(embs[query_idx], embs[i])) for i in doc_indices]
        ranked = sorted(sims, key=lambda x: -x[1])

        print(f"{variant} ranking:")
        for rank, (idx, sim) in enumerate(ranked[:5], 1):
            print(f"  #{rank} [{sim:.4f}] {test_sentences[idx][:60]}...")
        print()


Per-sentence cosine similarity vs FP32 baseline

BF16:
  ✓ 0.999940  •developments or disputes concerning our intellectual property or...
  ✓ 0.999923  We are subject to claims, lawsuits, regulatory and government inq...
  ✓ 0.999952  We are exposed to financial market risks, including changes in fo...
  ✓ 0.999945  Our marketable and non-marketable equity securities are subject t...
  ✓ 0.999946  Innovations in our products and services could also result in cha...
  ✓ 0.999944  From time to time, we acquire or invest in businesses, content, a...
  ✓ 0.999922  We view our employees and our culture as key to our success.  As ...
  ✓ 0.999944  As of December 27, 2025, we had approximately 31,000 employees in...
  ✓ 0.999927  Volatility in our stock price could adversely affect our business...
  ✓ 0.999918  The semiconductor industry is highly cyclical and has experienced...
  ✓ 0.999817  We believe trade dynamics and geopolitics are disrupting and resh...
  ✓ 0.999951  The “semiconductor

### Cell 9: Summary table

In [10]:
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
header = f"{'Variant':<8} {'VRAM (MB)':>10} {'Time (ms)':>14} {'Mean Sim':>10} {'Min Sim':>10} {'Δ VRAM':>8}"
print(header)
print("-" * 80)

fp32_vram = results["FP32"]["vram_mb"]

for variant in ["FP32", "BF16", "INT8", "NF4"]:  # Changed from FP16 to BF16
    res = results[variant]
    embs = res["embeddings"]

    if variant == "FP32":
        mean_sim, min_sim = 1.0, 1.0
    else:
        sims = [cosine_sim(fp32_embs[i], embs[i]) for i in range(len(fp32_embs))]
        mean_sim, min_sim = np.mean(sims), np.min(sims)

    vram_pct = (1 - res["vram_mb"] / fp32_vram) * 100 if fp32_vram > 0 else 0

    print(
        f"{variant:<8} {res['vram_mb']:>10.1f} "
        f"{res['mean_time_ms']:>8.1f} ± {res['std_time_ms']:>3.1f} "
        f"{mean_sim:>10.6f} {min_sim:>10.6f} "
        f"{'baseline' if variant == 'FP32' else f'-{vram_pct:.0f}%':>8}"
    )

print("-" * 80)
print("Sim = cosine similarity vs FP32 embeddings (1.000000 = identical)")



SUMMARY
Variant   VRAM (MB)      Time (ms)   Mean Sim    Min Sim   Δ VRAM
--------------------------------------------------------------------------------
FP32         1177.7    595.2 ± 73.2   1.000000   1.000000 baseline
BF16          592.0    774.8 ± 44.7   0.999926   0.999772     -50%
INT8          883.6    872.1 ± 52.4   0.999040   0.998652     -25%
NF4           840.8    447.1 ± 28.1   0.974997   0.961353     -29%
--------------------------------------------------------------------------------
Sim = cosine similarity vs FP32 embeddings (1.000000 = identical)
